# Episode 7: Routing Agents: Conditions, Handoffs & Escalation

In a hospital, the triage nurse doesn't treat you — they route you to the right specialist. Agent handoffs work the same way.

**By the end of this episode, you will:**
- Build a customer support triage system that routes to specialists
- Use `OnCondition` with LLM-based and expression-based conditions
- Control post-turn behavior with `set_after_work()`
- Understand when handoffs outperform auto-selection (and when they don't)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

## From smart selection to explicit routing

In Episode 6, an LLM decided who speaks next (AutoPattern). That's flexible, but it's also unpredictable — the LLM might route a billing question to your tech agent just because the customer mentioned a "dashboard."

**DefaultPattern with handoffs** flips that model. Instead of the LLM guessing, YOU define the routing rules. Each agent declares conditions under which it should hand off to another agent. The result is deterministic, auditable routing that you can reason about before you ever run the system.

There are two types of conditions. **LLM conditions** (`StringLLMCondition`) have an LLM evaluate a natural language prompt like *"The customer is asking about billing."* **Context conditions** (`ExpressionContextCondition`) check a variable directly, like `${severity} >= 7`. You can mix both in the same agent.

## Key concepts

Four building blocks make the handoff system work.

- **`OnCondition`** wraps a condition and a target: "IF this condition is true, THEN hand off to this agent"
- **`AgentTarget`** specifies which agent to hand off to
- **`RevertToUserTarget`** hands control back to the user (the conversation pauses so they can respond)
- **`set_after_work()`** defines what happens after an agent finishes its turn — the default behavior when no handoff condition fires

## Step 1: Create specialist agents

We need two specialists: one for billing, one for tech. Each gets a focused system message so it stays in its lane.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig
from autogen.agentchat.group import (
    AgentTarget,
    RevertToUserTarget,
    OnCondition,
    StringLLMCondition,
)

llm_config = LLMConfig({"model": "gpt-4o-mini"})

billing_agent = ConversableAgent(
    name="billing_agent",
    system_message=(
        "You are a billing specialist. Help customers with invoices, payments, "
        "refunds, and subscription questions. Be helpful and concise."
    ),
    description="Handles billing, payment, invoice, and subscription issues.",
    llm_config=llm_config, human_input_mode="NEVER",
)

tech_agent = ConversableAgent(
    name="tech_agent",
    system_message=(
        "You are a technical support specialist. Help customers with bugs, errors, "
        "setup issues, and technical questions. Provide step-by-step solutions."
    ),
    description="Handles technical issues, bugs, errors, and setup problems.",
    llm_config=llm_config, human_input_mode="NEVER",
)

## Step 2: Create a triage agent with handoff conditions

The triage agent is the front door. It greets the customer, reads their message, and fires the appropriate handoff. Let's see this:

In [ ]:
triage_agent = ConversableAgent(
    name="triage_agent",
    system_message=(
        "You are a customer support triage agent. Greet the customer, understand their issue, "
        "and route them to the right specialist. Do NOT try to solve the problem yourself."
    ),
    description="First point of contact. Routes customers to the right specialist.",
    llm_config=llm_config,
        human_input_mode="NEVER",
    handoffs=[
        OnCondition(
            target=AgentTarget(billing_agent),
            condition=StringLLMCondition(
                prompt="The customer is asking about billing, payments, invoices, refunds, or subscriptions."
            ),
        ),
        OnCondition(
            target=AgentTarget(tech_agent),
            condition=StringLLMCondition(
                prompt="The customer is asking about a technical issue, bug, error, or setup problem."
            ),
        ),
    ],
)

## Step 3: Set after_work behavior

After a specialist finishes, we want the conversation to pause so the customer can respond. `RevertToUserTarget()` does exactly that — it hands control back to the human in the loop.

In [ ]:
billing_agent.set_after_work(RevertToUserTarget())
tech_agent.set_after_work(RevertToUserTarget())

## Step 4: Run the triage system

Everything is wired up. Let's send in a billing question and watch the routing happen.

In [ ]:
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group.patterns import DefaultPattern

user = ConversableAgent(name="user", llm_config=llm_config, human_input_mode="NEVER")

pattern = DefaultPattern(
    initial_agent=triage_agent,
    agents=[triage_agent, billing_agent, tech_agent],
)

result, context, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Hi, I was charged twice for my subscription last month. Can you help?",
    max_rounds=5,
)

## What just happened?

Follow the trace in the output above. The **triage_agent** received the customer message and evaluated its handoff conditions. The LLM determined this is a billing issue, so the conversation was **handed off** to the **billing_agent**. After responding, the billing agent **reverted to the user** via `RevertToUserTarget()`.

Every step is explicit and auditable — you can trace exactly why each routing decision was made. In production e-commerce systems, this kind of deterministic routing has achieved 84% first-call resolution rates with 58% faster resolution times across 50K daily interactions.

Try changing the message to a technical issue — the triage agent will route to `tech_agent` instead.

## Trade-offs: Handoffs vs Auto

Handoffs shine when routing is predictable. If you know the categories upfront — billing, tech, general — explicit conditions give you reliability, auditability, and lower cost (no extra LLM call for speaker selection). They're a natural fit for customer support routing, compliance-critical systems, and any workflow where misrouting has a real cost.

AutoPattern is better when routing is genuinely unpredictable. If you're building an exploratory research assistant where the conversation could go in dozens of directions, writing handoff conditions for every path is brittle and exhausting. Let the LLM figure it out.

A good rule of thumb: if you can draw the routing diagram on a whiteboard, use handoffs. If the diagram would look like spaghetti, use Auto.

## Try it yourself

What happens when a customer asks something that doesn't fit billing or tech — like "What are your business hours?" Right now, the triage agent has no path for that. Can you add a `general_agent` with a third `OnCondition` to handle the catch-all case?

## Additional Content

### Context Variables

Think of context variables as a shared whiteboard that all agents can read from and write to. As the conversation flows through different agents, context variables carry state — routing decisions, intermediate results, accumulated facts — so each agent has the information it needs without re-asking the user.

```python
from autogen.agentchat.group import ContextVariables

# Create shared context
context = ContextVariables(data={"severity": 0, "category": "", "resolved": False})
```

Agents can access context variables in three ways. **Function parameters** let tool functions receive and update context directly. **UpdateSystemMessage** injects context values into system prompts dynamically, so an agent's behavior adapts as context changes. And **agent state** allows agents to read and write context programmatically.

Some patterns that work well in practice:

- Track routing state: `{"routed_to": "billing", "escalation_level": 1}`
- Store intermediate results: `{"diagnosis": "...", "recommendation": "..."}`
- Accumulate data across turns: `{"facts_gathered": ["fact1", "fact2"]}`

Two pitfalls to watch for. **Context pollution** happens when you store too much data, and later agents get confused by irrelevant information. **Missing initialization** means forgetting to set default values, which causes `KeyError` at runtime. Keep your context lean and always provide defaults.

### Escalation pattern: L1 -> L2 -> L3

You can chain handoffs to create escalation tiers. Here's the key part — using `ExpressionContextCondition` for confidence-based routing:

```python
from autogen.agentchat.group import ExpressionContextCondition

l1_agent = ConversableAgent(
    name="l1_support",
    system_message="You are L1 support. Handle simple questions. If unsure, set confidence to 0.",
    handoffs=[
        OnCondition(
            target=AgentTarget(l2_agent),
            condition=ExpressionContextCondition(
                expression="${confidence} < 50"
                ),
            ),
        ],
        human_input_mode="NEVER",
    )
```

This creates an automatic escalation path: if L1 isn't confident, the conversation escalates to L2. Chain another condition from L2 to L3, and you have a full tiered support system — no human dispatcher required.

### Triage with priorities

For more complex routing, you can separate classification from routing. A classifier agent determines both the category and priority, then handoff conditions route based on the classification. Here's what that looks like:

```python
classifier = ConversableAgent(
    name="classifier",
    system_message=(
        "Classify customer issues into: billing, technical, general. "
        "Also assign priority: low, medium, high, critical. "
        "Respond with: Category: <category>, Priority: <priority>"
    ),
    handoffs=[
        OnCondition(
            target=AgentTarget(billing_agent),
                condition=StringLLMCondition(
                    prompt="The classification indicates a billing issue."
                ),
            ),
            OnCondition(
                target=AgentTarget(tech_agent),
                condition=StringLLMCondition(
                    prompt="The classification indicates a technical issue."
                ),
            ),
        ],
        human_input_mode="NEVER",
    )
```

This two-stage approach — classify first, then route — gives you a clean separation of concerns. The classifier can be fine-tuned independently, and the routing logic stays simple.

## What's Next

You've built routing that's explicit and predictable. But what happens when agents need to process data in stages — like an assembly line? That's exactly what pipelines and hierarchies solve.

In **Episode 8**, you'll build **linear pipelines and hierarchical structures** — agents that process data in sequence or delegate to sub-teams.

**Bonus:** Try the handoff and context variable demos in the [AG2 Playground](https://playground.ag2.ai). Also see the [Intelligent Agent Handoffs blog post](https://ag2.ai/blog/intelligent-agent-handoffs).